# 15 — Phase 6 Integration Showcase

End-to-end demonstration combining profile registry, Tier 1/2/3 fidelity analysis, drift monitoring, and parallel streaming into a single workflow.

In [ ]:
from sqllocks_spindle import Spindle, ProfileRegistry, StreamingMultiWriter
from sqllocks_spindle.inference import AdvancedProfiler, DataProfiler, DatasetProfile
from sqllocks_spindle.inference.tier2_profiler import run_tier2
from sqllocks_spindle.inference.tier3_research import DriftMonitor, ChowLiuNetwork
from sqllocks_spindle.streaming import ConsoleSink, FileSink
from sqllocks_spindle.streaming.stream_writer import StreamWriter
from pathlib import Path
import importlib, io
from sqllocks_spindle.cli import _get_domain_registry

print('All Phase 6 imports OK')
spindle = Spindle()
reg_info = _get_domain_registry()
mod, cls, _ = reg_info['retail']
domain = getattr(importlib.import_module(mod), cls)(schema_mode='3nf')

In [ ]:
# Step 1: Generate reference data and save profiles
ref = spindle.generate(domain=domain, scale='small', seed=42)
profiler = DataProfiler(sample_rows=500)
reg = ProfileRegistry(root=Path('/tmp/integration_profiles'))
table_profiles = {n: profiler.profile(df, table_name=n) for n, df in ref.tables.items()}
saved = reg.save_from_dataset_profile(DatasetProfile(tables=table_profiles), system='retail', name='v1')
print(f'Step 1: Saved {len(saved)} profiles to registry')

In [ ]:
# Step 2: Generate new data
new = spindle.generate(domain=domain, scale='small', seed=99)
table = next(t for t in ref.tables if t in new.tables)
print(f'Step 2: Generated new data — validating table: {table}')

In [ ]:
# Step 3: Fidelity validation
report = reg.validate(saved[0].identity if saved else f'retail/{table}/v1', new)
print(f'Step 3: Fidelity score = {report.overall_score:.1f}/100')

In [ ]:
# Step 4: Advanced profiling
adv_profiler = AdvancedProfiler(max_gmm_components=2)
adv = adv_profiler.profile_pair(ref.tables[table], new.tables[table], table_name=table)
if adv.adversarial:
    print(f'Step 4: Adversarial AUC = {adv.adversarial.auc_roc:.3f}  (lower = more similar)')

In [ ]:
# Step 5: Tier 2 checks
t2 = run_tier2(ref.tables[table], new.tables[table])
print(f'Step 5: Tier 2 passing rate = {t2.passing_rate():.1%}')

In [ ]:
# Step 6: Drift monitoring
monitor = DriftMonitor()
drift = monitor.compare(ref.tables[table], new.tables[table])
print(f'Step 6: Drift fraction = {drift.drift_fraction:.1%}')

In [ ]:
# Step 7: Chow-Liu structure learning
cl = ChowLiuNetwork(n_bins=8)
cl_result = cl.fit(new.tables[table])
top_edge = max(cl_result.edges, key=lambda e: e.mutual_information, default=None)
if top_edge:
    print(f'Step 7: Strongest dependency: {top_edge.parent} → {top_edge.child}  MI={top_edge.mutual_information:.4f}')

In [ ]:
# Step 8: Parallel 4-sink streaming
class CountSink(StreamWriter):
    def __init__(self): self.count = 0
    def send_batch(self, events): self.count += len(events)

sinks = {f'sink_{i}': CountSink() for i in range(4)}
smw = StreamingMultiWriter(**sinks)
stream_result = smw.stream(spindle.generate_stream(domain=domain, scale='small', seed=0))
print(f'Step 8: Streaming to 4 sinks — {stream_result.total_tables} tables, success={stream_result.success}')

In [ ]:
# Final summary
print('\n' + '=' * 50)
print('Phase 6 Integration Complete')
print('=' * 50)
print(f'  Profiles saved:        {len(saved)}')
print(f'  Fidelity score:        {report.overall_score:.1f}/100')
if adv.adversarial:
    print(f'  Adversarial AUC:       {adv.adversarial.auc_roc:.3f}')
print(f'  Tier 2 pass rate:      {t2.passing_rate():.1%}')
print(f'  Drift fraction:        {drift.drift_fraction:.1%}')
print(f'  Streaming tables:      {stream_result.total_tables}')